# Package Imoport and Installation

In [1]:
%%capture
!pip install ultralytics
!pip install onnx
!pip install onnx_tf
!pip install matplotlib
!pip install protobuf==3.20.1
!pip install keras==2.11.0
!pip install tensorflow-estimator==2.11.0
!pip install tensorboard==2.11.0
!pip install torchinfo
!pip install torchview

In [2]:
%%capture
import os
import json, datetime
import gc
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from ultralytics import YOLO
from torchinfo import summary
from yolov8_EE_network import YOLOv8n_EE #implemented network leveraging ultralytics and torch utils
from torch_graph import model_graph #implemented graph service for demonstrating model's architecture
from dataset_loader import YoloTxtDataset

In [3]:
#dataset direcctories(note: we have uploaded our dataset on our organization's drive, you may change your direcctories)
ROOT_DIR= '/content/drive/MyDrive/Bicycle' #our dataset's directory
test_folder= os.path.join(ROOT_DIR, 'test/images/')
DATA_YAML= os.path.join(ROOT_DIR, 'data.yaml') #dataset yaml file

MODEL_YAML = "yolov8n_ee.yaml"
DEVICE= 'cuda' if torch.cuda.is_available() else 'cpu' #device declearation for model traning
IMG_SIZE= 640 #size of image and input tensors
BATCH_SIZE= 16
EPOCHS= 2
class_count = 1
exit_layers_count = 6

(Optional) **T4-GPU Availability Checking and Drive Mounting for Providing Dataset Access**

If you are running this code on a local computer or server you may pass this section.

In [4]:
from google.colab import drive
drive.mount('/content/drive/')

%cd /content/drive/MyDrive/EE using ultralitycs
!ls

Mounted at /content/drive/
/content/drive/MyDrive/EE using ultralitycs
'Copy of Copy of YOLOv8_EarlyExit.ipynb'
'Copy of YOLOv8_EarlyExit.ipynb'
 exit_log.jsonl
 __pycache__
 requirements.txt
 runs
 tensor.pt
 test.ipynb
 torch_graph.ipynb
 torch_graph.py
 yolo11n.pt
 yolo_early_exit_torchview.png
 YOLOv8_EarlyExit_CustmizedTrain.ipynb
 YOLOv8_EarlyExit.ipynb
 yolov8_EE_network.py
 yolov8n_ee.yaml
 yolov8n.pt


In [5]:
print("Torch Cuda - GPU informaiton:")
print("availablity: ", torch.cuda.is_available())
print("name: ", torch.cuda.get_device_name())
print("properties: ", torch.cuda.get_device_properties(device= 0))

Torch Cuda - GPU informaiton:
availablity:  True
name:  Tesla T4
properties:  _CudaDeviceProperties(name='Tesla T4', major=7, minor=5, total_memory=15095MB, multi_processor_count=40, uuid=bc882b65-2e6d-9a9c-dbe5-329b8b80b164, pci_bus_id=0, pci_device_id=4, pci_domain_id=0, L2_cache_size=4MB)


# **Dataset Loading**


In [6]:
print("Loading train split of dataset ...")
train_dataset = YoloTxtDataset(
    img_dir= os.path.join(ROOT_DIR, 'train/images/'),
    label_dir= os.path.join(ROOT_DIR, 'train/labels/')
)
print("Train dataset instances count: ", len(train_dataset))

print("Loading validation split of dataset ...")
valid_dataset = YoloTxtDataset(
    img_dir= os.path.join(ROOT_DIR, 'valid/images/'),
    label_dir= os.path.join(ROOT_DIR, 'valid/labels/')
)
print("Validation dataset instances count: ", len(valid_dataset))


Loading train split of dataset ...
Train dataset instances count:  71
Loading validation split of dataset ...
Validation dataset instances count:  21


# Early Exit Implementation in YOLOv8n

## Adding Early-Exit layers using ultralytics utilities

Here we will reimplement YOLOv8's architecture and will add 5 detect layers in backbone and neck aim to increase its infrence time and decrease its computational overheads while we are tending to maintaion accuracy's rates static as far as its possible.

In [7]:
#Build model and extract its summary
yolo_ee = YOLOv8n_EE(nc=class_count, model_yaml=MODEL_YAML).to(DEVICE)

summary(yolo_ee)

Layer (type:depth-idx)                        Param #
YOLOv8n_EE                                    --
├─Conv: 1-1                                   --
│    └─Conv2d: 2-1                            1,728
│    └─BatchNorm2d: 2-2                       128
│    └─SiLU: 2-3                              --
├─Conv: 1-2                                   --
│    └─Conv2d: 2-4                            73,728
│    └─BatchNorm2d: 2-5                       256
│    └─SiLU: 2-6                              --
├─C2f: 1-3                                    --
│    └─Conv: 2-7                              --
│    │    └─Conv2d: 3-1                       32,768
│    │    └─BatchNorm2d: 3-2                  512
│    │    └─SiLU: 3-3                         --
│    └─Conv: 2-8                              --
│    │    └─Conv2d: 3-4                       81,920
│    │    └─BatchNorm2d: 3-5                  256
│    │    └─SiLU: 3-6                         --
│    └─ModuleList: 2-9                       

In [ ]:
model_graph(
    netwrok= yolo_ee.model,
    expand_nested=False,
    graph_name="YOLO_EE_Graph",
)

## Trian YOLOv8 with new early exit layers

Loss function and Accuracy metrics

In [8]:
from torch.nn import functional as F

def calculate_loss(pred, labels):
    # Convert lists/tuples of tensors into a single tensor if possible
    if isinstance(pred, (list, tuple)):
        # If it's a list of identically-shaped tensors, stack along batch
        try:
            pred = torch.stack(pred, dim=0)  # (K, ...)
        except Exception:
            raise TypeError(f"pred is a list with non-uniform shapes: {[tuple(p.shape) for p in pred]}")

    if not isinstance(pred, torch.Tensor):
        raise TypeError(f"Expected Tensor for logits, got {type(pred)}")

    # For classification-style CE only:

    return F.cross_entropy(pred, labels), None

    # Common YOLO detect case: (B, C, N) or (B, C, H, W)
    raise ValueError(
        f"You’re passing a detection tensor {tuple(pred.shape)} into CE. "
        f"Use YOLOv8 detection loss (DFL/IoU/cls) instead."
    )

In [9]:
optimizer = torch.optim.SGD(yolo_ee.parameters(), lr=0.01, weight_decay = 0.001, momentum = 0.9)
optimizer

SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    lr: 0.01
    maximize: False
    momentum: 0.9
    nesterov: False
    weight_decay: 0.001
)

In [10]:
%%capture
from torch.utils.data import DataLoader

def yolo_collate(batch):
    imgs, targets = zip(*batch)         # imgs: tuple of (C,H,W), targets: tuple of (Ni,5)
    imgs = torch.stack(imgs, dim=0)     # (B,C,H,W)
    return imgs, list(targets)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    collate_fn=yolo_collate
)

In [ ]:
total_step = len(train_loader)

for epoch in range(EPOCHS):
    print(f"Epoch {epoch+1} ================")
    # n_batch = (n_sample - 1) // batch_size + 1
    yolo_ee.train()
    total_sample = 0
    total_loss = total_accuracy = 0
    layers_accuracy = []
    losses = []
    [layers_accuracy.append(0) for i in range(6)]
    with tqdm(total=total_step) as bar_step:
        for i, (images, labels) in enumerate(train_loader):
            # Move tensors to the configured device
            images = images.to(DEVICE)
            if isinstance(labels, tuple):
              labels = torch.tensor(labels, dtype=torch.float32)
            labels = labels[0].to(DEVICE)

            # Forward pass
            print("forward section")
            outputs = yolo_ee(images)
            print("optimizer zero grad")
            optimizer.zero_grad()
            print("loss and accuracy calculation")
            loss, accuracy = calculate_loss(outputs[5], labels)
            for j in range(len(accuracy)):
                layers_accuracy[j] += accuracy[j]
            total_sample += labels.size(0)

            print("Backward and optimize")
            # Backward and optimize
            for j in range(len(loss)):
                if j + 1 == len(loss):
                    loss[j].backward()
                else:
                    loss[j].backward(retain_graph = True)
            optimizer.step()

            print("torch.mps.empty_cache")
            del images, labels, outputs
            torch.mps.empty_cache() if torch.backends.mps.is_available() else torch.cuda.empty_cache()
            gc.collect()

            if (i + 1) % 3 == 0 or i + 1 == total_step + 1:
                bar_step.set_postfix({
                            'avg_loss': float(total_loss) / (i + 1),
                            'avg_accuracy': (sum(layers_accuracy)/3) / (i + 1),
                        })
                cur_n_batch = i % 3 + 1
                bar_step.update(cur_n_batch)

    for p in range(len(layers_accuracy)):
        accuracy1 = 100 * layers_accuracy[p] / total_sample
        print(f'Epoch {epoch+1}: Accuracy of Exit layer {p + 1} = {accuracy1:.2f}%')


    # Validation
    with torch.no_grad():
        correct = 0
        total = 0
        for images, labels in valid_dataset:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
            outputs = yolo_ee(images)
            _, predicted = torch.max(outputs[-1].data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            del images, labels, outputs

        print('Accuracy of the network on the {} validation images: {} %'.format(5000, 100 * correct / total))

## Test and Validation